# 3. Learned ratios and unbinned Asimov fits


    Train process-to-S and systematic-variation density ratios with the pinned
    NSBI toolkit, construct the interference model, and compare learned and
    analytical unbinned Asimov scans. Run notebooks 1–2 first.

    Here NSBI ratio estimation uses classifiers trained with cross entropy; it
    does not require a separate normalizing-flow density estimator because the
    S reference can be sampled directly.

### Runtime and persistent run

In Colab, choose **Runtime → Change runtime type → GPU** for notebooks 2–3.
The source checkout lives in the temporary runtime; **datasets, checkpoints,
workspace files, and results live in Google Drive**. Use the same `RUN_NAME`
and `MODE` in every notebook. Set `MODE = "smoke"` for a short workflow check;
the production default generates five million events **per sample**.
Use `CONFIG_OVERRIDES` to change a computational budget without editing source,
for example `{"quadrature_per_process": 500_000, "epochs": 150}`. Use the same
overrides in all five notebooks, and choose a new `RUN_NAME` when changing them.

This repository may be private. Grant Colab access when opening it from GitHub.
For the runtime download, either save a GitHub fine-grained token with **Contents:
read** access to this repository as the Colab secret `GITHUB_TOKEN`, or enter it
in the hidden prompt. The token is used only in a Python HTTP header and is not
written into files, Git remotes, shell arguments, or notebook output. Alternatively,
upload a repository ZIP to `/content/poodemo_source.zip`; no token is then needed.
An existing `/content/poodemo` checkout is reused. Delete that checkout to fetch
new source after updating the repository; your Drive run is separate.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "demo-v1"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    REPO_DIR = Path("/content/poodemo")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import tempfile
        import urllib.error
        import urllib.request
        import zipfile

        uploaded_zip = Path("/content/poodemo_source.zip")
        if uploaded_zip.exists():
            archive_bytes = uploaded_zip.read_bytes()
            archive_kind = "zip"
        else:
            archive_url = "https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/main"

            def fetch_source(token=None):
                headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
                if token:
                    headers["Authorization"] = "Bearer " + token
                request = urllib.request.Request(archive_url, headers=headers)
                with urllib.request.urlopen(request, timeout=180) as response:
                    return response.read()

            try:
                archive_bytes = fetch_source()
            except urllib.error.HTTPError as error:
                if error.code not in (401, 403, 404):
                    raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
                token = None
                try:
                    from google.colab import userdata
                    token = userdata.get("GITHUB_TOKEN")
                except Exception:
                    pass
                if not token:
                    token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
                if not token:
                    raise RuntimeError("Upload /content/poodemo_source.zip or provide read access.")
                try:
                    archive_bytes = fetch_source(token)
                except urllib.error.HTTPError as auth_error:
                    raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
                finally:
                    token = None
            archive_kind = "tar"

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            if archive_kind == "zip":
                with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                    for entry in archive.infolist():
                        safe_destination(entry.filename)
                        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                            raise RuntimeError("Symbolic links are not supported in the source ZIP.")
                    archive.extractall(staging)
            else:
                with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                    for entry in archive.getmembers():
                        safe_destination(entry.name)
                        if not (entry.isfile() or entry.isdir()):
                            raise RuntimeError("Links and special files are not supported in the source archive.")
                    archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The uploaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    import subprocess
    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))

### Density ratios and the physical model

For the fixed selected phase space, set \(q=p_S^{\rm sel}\), and learn
\(r_j=p_j^{\rm sel}/q\) for the other processes. Shape ratios and physical
yields are distinct. At nominal nuisance parameters,
\[
\frac{D(x;\mu)}{q(x)}
=(\mu-\sqrt\mu)\lambda_S
 +\sqrt\mu\lambda_{SBI}r_{SBI}(x)
 +(1-\sqrt\mu)\lambda_Br_B(x)
 +\lambda_{NI}r_{NI}(x),
\]
where every \(\lambda\) is now a **selected** yield.

We also train up/down variation-to-nominal ratios. For B variations,
SBI varies coherently; NI has its own nuisance. The nuisance interpolation
is the toolkit's exponential–polynomial interpolation, with Gaussian
auxiliary constraints. An interpolated nuisance model is a chosen
statistical model; it need not equal a continuously shifted Gaussian
amplitude away from its nominal and ±1 anchors.

In [ ]:
from poodemo.pipeline import train_ratios

ratio_diagnostics = train_ratios(run)
display(ratio_diagnostics)

### Validate ratios before interpreting the inference

The analytical density is available in this toy. Held-out diagnostics
distinguish finite training errors from score compression and histogram
errors. Ratio normalization alone does not guarantee that the ratio is
correct point by point. Signed interference coefficients make numerical
positivity of the total learned intensity a particularly important
diagnostic; a failed physical-domain check must not be silently hidden
by clipping the final likelihood.

To isolate neural-network shape errors in this demonstration, the
learned and analytical likelihoods use the same accepted component
yields from the common analytical quadrature. Sample-estimated selected
rates are reported separately. Learned shapes are numerically normalized
on that quadrature after independent calibration, and their raw
integrals are saved. This uses the toy's known density for validation
and rate control; an analysis without an oracle would estimate those
quantities from its simulation and auxiliary measurements.

In [ ]:
from poodemo.pipeline import prepare_quadrature

quadrature = prepare_quadrature(run)
print('Quadrature products:', list(quadrature))

### Asimov integration and profiling

We approximate expectations with a held-out weighted integration sample.
These are Asimov likelihood curves, not a coverage study with random
Poisson pseudo-experiments. The extended likelihood retains the total
expected yield and includes the nuisance auxiliary measurements.

At the generating point \((\mu_*,\boldsymbol\alpha_*)=(1,\boldsymbol0)\),
an exact intensity model gives
\[
t_A(\mu,\boldsymbol\alpha)=2\int\!dx\,
\left[D(x;\mu,\boldsymbol\alpha)-D_*(x)
-D_*(x)\log\frac{D(x;\mu,\boldsymbol\alpha)}{D_*(x)}\right]
+\|\boldsymbol\alpha\|^2,
\]
followed by profiling. Finite quadrature and learned-ratio errors are
separate effects. The pipeline saves both analytical and learned
reference curves so that later comparisons can identify their origin.

In [ ]:
from poodemo.pipeline import run_unbinned

unbinned = run_unbinned(run)
display(unbinned['scans'])

In [ ]:
def plot_scans(frame, title, filename, group_columns=None):
    """Plot saved scan tables; adapt grouping to the columns present."""
    x_column = next((c for c in ["mu", "mu_test", "mu0"] if c in frame.columns), None)
    y_column = next((c for c in ["q", "t_mu", "test_statistic", "ts"] if c in frame.columns), None)
    if x_column is None or y_column is None:
        raise ValueError(f"Expected mu and q scan columns; found {list(frame.columns)}")
    if group_columns is None:
        group_columns = [c for c in ["method", "model", "systematics", "nuisances", "n_bins", "bins", "observable"] if c in frame.columns]
    groups = frame.groupby(group_columns, dropna=False, sort=False) if group_columns else [("scan", frame)]
    fig, ax = plt.subplots()
    for label, group in groups:
        group = group.sort_values(x_column)
        values = label if isinstance(label, tuple) else (label,)
        parts = []
        for column, value in zip(group_columns, values):
            if pd.isna(value):
                continue
            if column == "systematics":
                parts.append("profiled" if value else "stat only")
            elif column in ("n_bins", "bins"):
                parts.append(f"{int(value)} bins")
            else:
                parts.append(str(value))
        ax.plot(group[x_column], group[y_column], marker=".", ms=3, label=" | ".join(parts))
    ax.axhline(1.0, color="0.45", ls=":", lw=1)
    ax.set(xlabel=r"Tested signal strength $\mu_0$", ylabel=r"$t(\mu_0)$", title=title)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8)
    fig.tight_layout()
    output = ROOT / "plots"
    output.mkdir(exist_ok=True)
    fig.savefig(output / filename, bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
plot_scans(unbinned['scans'], 'Unbinned Asimov likelihood scans', '03_unbinned_scans.pdf');

### Check the numerical expectation itself

The next diagnostic evaluates the nominal-nuisance analytical Asimov
objective using the full integration sample and two disjoint halves.
The split is made **within each proposal stratum before selection**;
rejected events contribute zero to the uncertainty calculation.
Error bars estimate Monte Carlo integration uncertainty, not statistical
confidence intervals on \(\mu\). Half-sample agreement and small
estimated errors are checks on the numerical expectation; a precision
study should also increase the integration budget in a separate run.

In [ ]:
integration_check = unbinned["quadrature"].sort_values("mu")
display(integration_check)
fig, ax = plt.subplots()
ax.errorbar(integration_check["mu"], integration_check["q"],
            yerr=integration_check["standard_error"], fmt="o-", ms=3,
            capsize=2, label="Full sample ± integration SE")
ax.plot(integration_check["mu"], integration_check["q_half_even"], "--", label="Even half")
ax.plot(integration_check["mu"], integration_check["q_half_odd"], ":", label="Odd half")
ax.set(xlabel=r"Signal strength $\mu$", ylabel="Asimov likelihood objective",
       title="Integration stability: fixed nominal nuisances")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(ROOT / "plots" / "03_quadrature_stability.pdf", bbox_inches="tight")
plt.show()

In [ ]:
for key, value in unbinned.items():
    if key not in ("scans", "quadrature"):
        print(key)
        display(value)

The horizontal line at \(t=1\) is a visual likelihood-scale reference.
These Asimov curves alone do not demonstrate interval coverage or
establish Wilks' theorem at a boundary. The scan uses positive signal
strengths; the derivative of the \(\sqrt\mu\) model is singular at zero.
The default scan is \(0.1\le\mu\le3\); the fit domain is
\(0.001\le\mu\le4\). The lower fit bound is a numerical floor for
this regular interior demonstration, not an implementation of a
discovery test at the physical boundary \(\mu=0\).

Short smoke training may give visibly inaccurate learned curves or
invalid learned intensities. Such points are flagged and appear as gaps;
they do not count as successful learned-model closure. Inspect fit
validity and ratio diagnostics before increasing the training budget.

Next: **04_score_histograms.ipynb**.